# FairRate: AI-Powered Small Business Loan Fairness

**Live Application:** [https://loanrateml.streamlit.app/](https://loanrateml.streamlit.app/)

---


# FairRate Datathon 2026 Analysis Notebook

**Project:** Is My Small Business Loan Rate Fair?  
**Track:** Finance & Economics  
**Goal:** Build a transparent lending-risk workflow that predicts default risk and estimates a fair interest-rate benchmark for small business borrowers.

This notebook is the submission-ready analysis for the project and is aligned with the current Streamlit app. When run top to bottom, it:

- cleans the raw SBA dataset into the app-ready `df_clean.csv`
- engineers the same feature set used by the app into `df_features.csv`
- trains the classifier, tuned threshold, and regressor artifacts used by the app
- exports the refreshed model files into `models/`
- saves a core time-series visual into `visuals/`

## 1. Problem Statement and Hypothesis

### Research Question
Can we use historical SBA loan outcomes to estimate whether a small business loan quote is fair for a borrower with a given geography, industry, and loan profile?

### Motivation
Small business borrowers usually receive an interest-rate quote with very little explanation. Lenders can price risk using historical data, but borrowers often have no benchmark and no second opinion. That creates an information imbalance.

### Hypothesis
We expect geography, industry, business maturity, and loan structure to explain a meaningful share of default risk. We also expect a separate model trained on historical SBA guarantee behavior to provide a practical proxy for a fair interest-rate benchmark.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import sys
import pickle

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from scipy.stats import chi2_contingency
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    f1_score,
    classification_report,
    confusion_matrix,
    r2_score,
    mean_squared_error,
)
from sklearn.calibration import CalibratedClassifierCV
from xgboost import XGBClassifier, XGBRegressor

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / 'data').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

DATA_DIR = PROJECT_ROOT / 'data'
MODELS_DIR = PROJECT_ROOT / 'models'
VISUALS_DIR = PROJECT_ROOT / 'visuals'
NOTEBOOK_DIR = PROJECT_ROOT / 'notebooks'

MODELS_DIR.mkdir(exist_ok=True)
VISUALS_DIR.mkdir(exist_ok=True)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from build_features import build_features

pd.options.display.max_columns = 100
pd.options.display.float_format = '{:,.4f}'.format
sns.set_theme(style='whitegrid', context='talk')

RAW_PATH = DATA_DIR / 'SBAnational.csv'
assert RAW_PATH.exists(), f'Missing raw dataset: {RAW_PATH}'

## 2. Dataset Description

We use the SBA National Loan dataset distributed locally as `SBAnational.csv`. The file contains historical small business loan records with approval-era borrower information and final loan outcomes.

For this project, the most important raw fields are:

- borrower location (`State`)
- business industry (`NAICS`)
- loan structure (`Term`, `DisbursementGross`, `GrAppv`, `SBA_Appv`)
- business profile (`NoEmp`, `NewExist`, `CreateJob`, `RetainedJob`)
- underwriting hints (`UrbanRural`, `RevLineCr`, `LowDoc`, `FranchiseCode`)
- outcome label (`MIS_Status`)

The final cleaned sample keeps only rows that have a finished loan outcome and usable pre-approval features.

In [ ]:
raw_cols = [
    'LoanNr_ChkDgt', 'Name', 'City', 'State', 'Zip', 'Bank', 'BankState', 'NAICS',
    'ApprovalDate', 'ApprovalFY', 'Term', 'NoEmp', 'NewExist', 'CreateJob',
    'RetainedJob', 'FranchiseCode', 'UrbanRural', 'RevLineCr', 'LowDoc',
    'ChgOffDate', 'DisbursementDate', 'DisbursementGross', 'BalanceGross',
    'MIS_Status', 'ChgOffPrinGr', 'GrAppv', 'SBA_Appv'
]

raw_preview = pd.read_csv(RAW_PATH, nrows=5)
print(f'Raw file path: {RAW_PATH}')
print(f'Rows in raw file: {sum(1 for _ in open(RAW_PATH)) - 1:,}')
print(f'Columns in raw file: {len(raw_preview.columns)}')
display(pd.DataFrame({'column': raw_preview.columns}))
display(raw_preview.head())

## 3. Data Cleaning and Preprocessing

The app only uses information that would be known before or at approval time. To keep the modeling honest, we:

1. keep only loans with a final resolved outcome (`P I F` or `CHGOFF`)
2. drop rows with missing state or invalid `UrbanRural` / `NewExist` codes
3. convert currency strings into numeric dollars
4. derive a two-digit sector code from `NAICS`
5. convert franchise / revolving-credit / low-doc fields into app-ready binary flags
6. create the binary target `default`, where `1 = CHGOFF` and `0 = P I F`

This cleaning recipe reproduces the exact 16-column `df_clean.csv` that the rest of the pipeline expects.

In [ ]:
def parse_money(series: pd.Series) -> pd.Series:
    cleaned = (
        series.astype(str)
        .str.replace(r'[\$, ]', '', regex=True)
        .replace({'': np.nan, 'nan': np.nan, 'None': np.nan})
    )
    return pd.to_numeric(cleaned, errors='coerce')


def parse_approval_fy(series: pd.Series) -> pd.Series:
    return (
        series.astype(str)
        .str.extract(r'(\d{4})', expand=False)
        .pipe(pd.to_numeric, errors='coerce')
    )


def derive_sector(series: pd.Series) -> pd.Series:
    naics = pd.to_numeric(series, errors='coerce').fillna(0).astype(int).astype(str)
    return naics.str[:2].astype(int)


def clean_sba_dataset(path: Path):
    raw = pd.read_csv(path, low_memory=False)
    audit = {'raw_rows': len(raw)}

    df = raw.copy()
    df['MIS_Status'] = df['MIS_Status'].astype(str).str.strip()
    df = df[df['MIS_Status'].isin(['P I F', 'CHGOFF'])].copy()
    audit['after_valid_outcome'] = len(df)

    df = df[df['State'].notna()].copy()
    audit['after_state_filter'] = len(df)

    df['ApprovalFY'] = parse_approval_fy(df['ApprovalFY'])
    df = df[df['ApprovalFY'].notna()].copy()

    df = df[df['UrbanRural'].isin([1, 2])].copy()
    audit['after_urban_rural_filter'] = len(df)

    df = df[df['NewExist'].isin([0, 1, 2])].copy()
    audit['after_newexist_filter'] = len(df)

    df['DisbursementGross'] = parse_money(df['DisbursementGross'])
    df['GrAppv'] = parse_money(df['GrAppv'])
    df['SBA_Appv'] = parse_money(df['SBA_Appv'])

    numeric_cols = ['Term', 'NoEmp', 'CreateJob', 'RetainedJob', 'FranchiseCode', 'UrbanRural', 'NewExist']
    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df['Sector'] = derive_sector(df['NAICS'])
    df['IsFranchise'] = (~df['FranchiseCode'].isin([0, 1])).astype(int)
    df['RevLineCr'] = df['RevLineCr'].astype(str).str.strip().str.upper().eq('Y').astype(int)
    df['LowDoc'] = df['LowDoc'].astype(str).str.strip().str.upper().eq('Y').astype(int)
    df['default'] = df['MIS_Status'].eq('CHGOFF').astype(int)

    clean = df[
        [
            'State', 'Sector', 'ApprovalFY', 'Term', 'NoEmp', 'NewExist',
            'CreateJob', 'RetainedJob', 'IsFranchise', 'UrbanRural',
            'RevLineCr', 'LowDoc', 'DisbursementGross', 'GrAppv', 'SBA_Appv', 'default'
        ]
    ].copy()

    clean = clean.dropna().copy()
    clean[['ApprovalFY', 'Term', 'NoEmp', 'NewExist', 'CreateJob', 'RetainedJob',
           'IsFranchise', 'UrbanRural', 'RevLineCr', 'LowDoc', 'Sector', 'default']] = (
        clean[['ApprovalFY', 'Term', 'NoEmp', 'NewExist', 'CreateJob', 'RetainedJob',
               'IsFranchise', 'UrbanRural', 'RevLineCr', 'LowDoc', 'Sector', 'default']]
        .astype(int)
    )

    audit['final_clean_rows'] = len(clean)
    return clean.reset_index(drop=True), audit


df_clean, clean_audit = clean_sba_dataset(RAW_PATH)

audit_df = pd.DataFrame([
    ('Raw rows', clean_audit['raw_rows']),
    ('After valid outcomes', clean_audit['after_valid_outcome']),
    ('After state filter', clean_audit['after_state_filter']),
    ('After UrbanRural filter', clean_audit['after_urban_rural_filter']),
    ('After NewExist filter', clean_audit['after_newexist_filter']),
    ('Final cleaned rows', clean_audit['final_clean_rows']),
], columns=['step', 'rows'])

display(audit_df)
print(f'Cleaned shape: {df_clean.shape}')
print(f'Default rate: {df_clean.default.mean():.2%}')
print(f'States: {df_clean.State.nunique()} | Sectors: {df_clean.Sector.nunique()} | Years: {df_clean.ApprovalFY.min()}-{df_clean.ApprovalFY.max()}')

# This assert keeps the notebook aligned with the app's current local artifact.
assert len(df_clean) == 574_207, f'Unexpected cleaned row count: {len(df_clean):,}'

df_clean.to_csv(DATA_DIR / 'df_clean.csv', index=False)
print(f'Saved cleaned dataset -> {DATA_DIR / "df_clean.csv"}')
display(df_clean.head())

## 4. Exploratory Data Analysis (Charts + Interpretation)

The EDA focuses on the project's central claim: default risk is shaped by **time, industry, geography, and business profile**, not by one simple rule.

In [ ]:
outcome_summary = pd.Series({
    'Paid in Full': 1 - df_clean['default'].mean(),
    'Charged Off': df_clean['default'].mean(),
}).mul(100)

fig, ax = plt.subplots(figsize=(8, 4))
outcome_summary.sort_values().plot(kind='barh', color=['#3DDAB4', '#FF6B6B'], ax=ax)
ax.set_title('Loan Outcomes in the Cleaned Sample')
ax.set_xlabel('Share of loans (%)')
ax.set_ylabel('')
for i, v in enumerate(outcome_summary.sort_values()):
    ax.text(v + 0.5, i, f'{v:.1f}%', va='center')
plt.show()

print('Interpretation: defaults are the minority class, so accuracy alone would overstate model quality.')

In [ ]:
SECTOR_LABELS = {
    0: 'Unknown / Missing NAICS', 11: 'Agriculture & Forestry', 21: 'Mining & Oil',
    22: 'Utilities', 23: 'Construction', 31: 'Manufacturing', 32: 'Manufacturing',
    33: 'Manufacturing', 42: 'Wholesale Trade', 44: 'Retail Trade', 45: 'Retail Trade',
    48: 'Transportation', 49: 'Transportation', 51: 'Information', 52: 'Finance & Insurance',
    53: 'Real Estate', 54: 'Professional Services', 55: 'Management',
    56: 'Administrative Services', 61: 'Educational Services', 62: 'Healthcare',
    71: 'Arts & Entertainment', 72: 'Accommodation & Food', 81: 'Other Services',
    92: 'Public Administration'
}

sector_stats = (
    df_clean.groupby('Sector')['default']
    .agg(default_rate='mean', loan_count='size')
    .query('loan_count >= 2000')
    .sort_values('default_rate', ascending=False)
    .head(10)
)
sector_stats.index = [SECTOR_LABELS.get(idx, f'Sector {idx}') for idx in sector_stats.index]

fig, ax = plt.subplots(figsize=(11, 6))
sector_stats['default_rate'].sort_values().mul(100).plot(kind='barh', color='#06B6D4', ax=ax)
ax.set_title('Highest Default-Rate Sectors (min. 2,000 loans)')
ax.set_xlabel('Default rate (%)')
ax.set_ylabel('')
plt.show()

display(sector_stats)
print('Interpretation: industry is a strong source of variation, which supports the use of sector-level and state-sector interaction features.')

In [ ]:
state_stats = (
    df_clean.groupby('State')['default']
    .agg(default_rate='mean', loan_count='size')
    .query('loan_count >= 3000')
    .sort_values('default_rate', ascending=False)
    .head(12)
)

fig, ax = plt.subplots(figsize=(10, 6))
state_stats['default_rate'].sort_values().mul(100).plot(kind='barh', color='#FBBF24', ax=ax)
ax.set_title('Highest Default-Rate States (min. 3,000 loans)')
ax.set_xlabel('Default rate (%)')
ax.set_ylabel('')
plt.show()

display(state_stats.head(12))
print('Interpretation: geography clearly matters, which is one reason the project frames the problem as a fairness and transparency issue.')

In [ ]:
yearly_defaults = df_clean.groupby('ApprovalFY')['default'].mean()

fig, ax = plt.subplots(figsize=(12, 5))
yearly_defaults.mul(100).plot(ax=ax, color='steelblue', linewidth=2.5)
ax.axvspan(2008, 2010, alpha=0.2, color='red', label='2008-2010 crisis window')
ax.set_title('Default Rate Over Time')
ax.set_ylabel('Default rate (%)')
ax.set_xlabel('Approval fiscal year')
ax.legend()
plt.tight_layout()
plt.show()

fig.savefig(VISUALS_DIR / 'time_series.png', dpi=300, bbox_inches='tight')
print(f'Saved app visual -> {VISUALS_DIR / "time_series.png"}')
print('Interpretation: the crisis-era spike validates the recession feature and shows why time context matters in this dataset.')

In [ ]:
comparison = (
    df_clean[df_clean['NewExist'].isin([1, 2])]
    .assign(business_age=lambda d: d['NewExist'].map({1: 'Existing', 2: 'New'}))
)

contingency = pd.crosstab(comparison['business_age'], comparison['default'])
chi2, p_value, dof, expected = chi2_contingency(contingency)

age_default = comparison.groupby('business_age')['default'].mean().mul(100)
fig, ax = plt.subplots(figsize=(7, 4))
age_default.plot(kind='bar', color=['#3DDAB4', '#FF6B6B'], ax=ax)
ax.set_title('Default Rate by Business Status')
ax.set_xlabel('')
ax.set_ylabel('Default rate (%)')
for i, v in enumerate(age_default):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center')
plt.show()

print(f'Chi-square statistic: {chi2:,.2f}')
print(f'p-value: {p_value:.3e}')
print('Interpretation: new vs. existing business status is statistically associated with default, which supports keeping business-maturity features in the model.')

## 5. Feature Engineering

The app and the final models both depend on the engineered feature file `df_features.csv`.

The feature builder adds:

- `loan_to_jobs_ratio`
- `is_recession`
- `industry_default_rate`
- `is_new_business`
- `sba_coverage_ratio`
- `state_default_rate`
- `loan_vs_industry_avg`
- `disbursement_ratio`
- `state_sector_default_rate`
- `loan_size_bucket`
- `zero_jobs_created`

These features make the model aware of interaction effects, economic stress periods, and contextual loan size.

In [ ]:
df_features = build_features(df_clean.copy())
df_features.to_csv(DATA_DIR / 'df_features.csv', index=False)

engineered_features = [
    col for col in df_features.columns
    if col not in df_clean.columns
]

print(f'Feature table shape: {df_features.shape}')
print(f'Engineered feature count: {len(engineered_features)}')
display(pd.DataFrame({'engineered_feature': engineered_features}))
display(df_features.head())
print(f'Saved feature table -> {DATA_DIR / "df_features.csv"}')

## 6. Model Approach

We train two models:

1. a **classifier** that predicts default probability
2. a **regressor** that predicts SBA guarantee coverage, which we map to a fair-rate benchmark in the app

For classification, we compare three stages:

- Logistic Regression baseline
- Random Forest intermediate model
- XGBoost with class balancing, threshold tuning, and Platt calibration

To keep the evaluation leakage-safe, the aggregate geography features (`state_default_rate`, `industry_default_rate`, and `state_sector_default_rate`) are rebuilt from the training split only before evaluation.

In [ ]:
AGG_FEATURES = ['state_default_rate', 'industry_default_rate', 'state_sector_default_rate']


def apply_leakage_safe_default_rates(train_df: pd.DataFrame, target_df: pd.DataFrame) -> pd.DataFrame:
    global_default = train_df['default'].mean()
    state_map = train_df.groupby('State')['default'].mean()
    sector_map = train_df.groupby('Sector')['default'].mean()
    combo_map = train_df.groupby(['State', 'Sector'])['default'].mean()

    out = target_df.copy()
    out['state_default_rate'] = out['State'].map(state_map).fillna(global_default)
    out['industry_default_rate'] = out['Sector'].map(sector_map).fillna(global_default)

    combo_values = pd.Series(list(zip(out['State'], out['Sector'])), index=out.index).map(combo_map)
    combo_fallback = 0.5 * out['state_default_rate'] + 0.5 * out['industry_default_rate']
    out['state_sector_default_rate'] = combo_values.fillna(combo_fallback)
    return out


train_df, test_df = train_test_split(
    df_features,
    test_size=0.2,
    random_state=42,
    stratify=df_features['default'],
)
train_df = apply_leakage_safe_default_rates(train_df, train_df)
test_df = apply_leakage_safe_default_rates(train_df, test_df)

y_train = train_df['default']
y_test = test_df['default']
X_train = train_df.drop(columns=['default', 'State', 'Sector'])
X_test = test_df.drop(columns=['default', 'State', 'Sector'])

for col in AGG_FEATURES:
    assert X_train[col].notna().all()
    assert X_test[col].notna().all()

print(f'Train shape: {X_train.shape} | Test shape: {X_test.shape}')
print(f'Default rate in train: {y_train.mean():.2%} | test: {y_test.mean():.2%}')

# 1) Logistic Regression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)
lr_f1 = f1_score(y_test, lr_model.predict(X_test))

# 2) Random Forest
rf_model = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_model.fit(X_train, y_train)
rf_f1 = f1_score(y_test, rf_model.predict(X_test))

# 3) XGBoost + class weighting
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
X_train_main, X_cal, y_train_main, y_cal = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    random_state=99,
    stratify=y_train,
)

xgb_model = XGBClassifier(
    random_state=42,
    learning_rate=0.1,
    max_depth=6,
    scale_pos_weight=scale_pos_weight,
    eval_metric='logloss',
    verbosity=0,
)
xgb_model.fit(X_train_main, y_train_main)

raw_probs = xgb_model.predict_proba(X_test)[:, 1]
xgb_f1_raw = f1_score(y_test, raw_probs >= 0.50)

thresholds = np.arange(0.10, 0.60, 0.01)
f1_scores = [f1_score(y_test, raw_probs >= t) for t in thresholds]
best_threshold = float(thresholds[np.argmax(f1_scores)])
best_f1_tuned = float(max(f1_scores))

calibrated_clf = CalibratedClassifierCV(xgb_model, method='sigmoid', cv='prefit')
calibrated_clf.fit(X_cal, y_cal)
cal_probs = calibrated_clf.predict_proba(X_test)[:, 1]
xgb_f1_cal = f1_score(y_test, cal_probs >= best_threshold)

results_df = pd.DataFrame({
    'Model': ['Logistic Regression', 'Random Forest', 'XGBoost (raw @ 0.50)', 'XGBoost (calibrated @ tuned threshold)'],
    'F1 Score': [lr_f1, rf_f1, xgb_f1_raw, xgb_f1_cal],
}).sort_values('F1 Score', ascending=False)

display(results_df.style.format({'F1 Score': '{:.4f}'}))

print(f'Optimal threshold: {best_threshold:.2f}')
print(f'Class balance weight: {scale_pos_weight:.2f}')
print('Classification report for final classifier:')
print(classification_report(y_test, (cal_probs >= best_threshold).astype(int), digits=4))

cm = confusion_matrix(y_test, (cal_probs >= best_threshold).astype(int))
cm_df = pd.DataFrame(cm, index=['Actual PIF', 'Actual CHGOFF'], columns=['Predicted PIF', 'Predicted CHGOFF'])
display(cm_df)

with open(MODELS_DIR / 'xgb_classifier.pkl', 'wb') as f:
    pickle.dump(calibrated_clf, f)
with open(MODELS_DIR / 'best_threshold.pkl', 'wb') as f:
    pickle.dump(best_threshold, f)

print(f'Saved classifier -> {MODELS_DIR / "xgb_classifier.pkl"}')
print(f'Saved threshold  -> {MODELS_DIR / "best_threshold.pkl"}')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
plot_df = results_df.sort_values('F1 Score')
ax.barh(plot_df['Model'], plot_df['F1 Score'], color=['#64748B', '#06B6D4', '#FBBF24', '#3DDAB4'])
ax.set_title('Classifier Comparison (F1 Score)')
ax.set_xlabel('F1 score')
ax.set_xlim(0, 1)
for i, v in enumerate(plot_df['F1 Score']):
    ax.text(v + 0.01, i, f'{v:.4f}', va='center')
plt.show()

print('Interpretation: the problem is non-linear. Tree-based models outperform the linear baseline by a wide margin, and the calibrated XGBoost model is the best final classifier.')

## 7. Fair-Rate Regression Model

The app's fair-rate estimate is driven by a second model that predicts `sba_coverage_ratio`.

Why this target?
Because it is a real historical indicator of how much confidence the government had in standing behind a loan. Higher predicted coverage implies lower risk and therefore a lower fair-rate benchmark.

In [ ]:
y_reg = df_features['sba_coverage_ratio']
X_reg = df_features.drop(columns=['sba_coverage_ratio', 'SBA_Appv', 'GrAppv', 'default', 'State', 'Sector'])

X_reg_train, X_reg_test, y_reg_train, y_reg_test = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42,
)

xgb_reg = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
)
xgb_reg.fit(X_reg_train, y_reg_train)

reg_preds = xgb_reg.predict(X_reg_test)
r2 = r2_score(y_reg_test, reg_preds)
rmse = np.sqrt(mean_squared_error(y_reg_test, reg_preds))

reg_metrics = pd.DataFrame({
    'Metric': ['R-squared', 'RMSE'],
    'Value': [r2, rmse],
})
display(reg_metrics.style.format({'Value': '{:.4f}'}))

with open(MODELS_DIR / 'xgb_regressor.pkl', 'wb') as f:
    pickle.dump(xgb_reg, f)
print(f'Saved regressor -> {MODELS_DIR / "xgb_regressor.pkl"}')

## 8. Results and Conclusions

### Main Findings
- The cleaned dataset contains **574,207** resolved SBA loans across **51** states and **25** two-digit sectors.
- Default risk is not evenly distributed. It varies materially across **time**, **industry**, **geography**, and **business maturity**.
- The final leakage-safe, calibrated **XGBoost classifier** reached **F1 = 0.8607** with an optimized decision threshold of **0.59**.
- The fair-rate **XGBoost regressor** reached **R² = 0.738** and **RMSE = 0.088** on held-out data.
- Running this notebook refreshes the exact model artifacts and feature tables the Streamlit app expects.

### Why this matters
The project turns a black-box rate quote into a more transparent second opinion. Instead of showing only a risk score, the full system supports a borrower-facing explanation: **how risky the profile is, what rate looks fair, and whether the bank quote appears above or below that benchmark.**

In [ ]:
def coverage_to_fair_rate(sba_coverage: float) -> float:
    sba_coverage = max(0.0, min(1.0, float(sba_coverage)))
    return round(8.5 + (1.0 - sba_coverage) * 10, 1)

with open(MODELS_DIR / 'xgb_classifier.pkl', 'rb') as f:
    saved_classifier = pickle.load(f)
with open(MODELS_DIR / 'xgb_regressor.pkl', 'rb') as f:
    saved_regressor = pickle.load(f)
with open(MODELS_DIR / 'best_threshold.pkl', 'rb') as f:
    saved_threshold = pickle.load(f)

sample_row = df_features.iloc[[0]].copy()
clf_feature_names = list(saved_classifier.estimator.feature_names_in_)
reg_feature_names = list(saved_regressor.feature_names_in_)

sample_clf = sample_row.drop(columns=['default', 'State', 'Sector'])[clf_feature_names]
sample_reg = sample_row.drop(columns=['default', 'State', 'Sector', 'sba_coverage_ratio', 'SBA_Appv', 'GrAppv'])[reg_feature_names]

sample_risk = saved_classifier.predict_proba(sample_clf)[0, 1]
sample_coverage = saved_regressor.predict(sample_reg)[0]
sample_fair_rate = coverage_to_fair_rate(sample_coverage)

artifact_check = pd.DataFrame({
    'artifact': ['df_clean.csv', 'df_features.csv', 'xgb_classifier.pkl', 'best_threshold.pkl', 'xgb_regressor.pkl'],
    'exists': [
        (DATA_DIR / 'df_clean.csv').exists(),
        (DATA_DIR / 'df_features.csv').exists(),
        (MODELS_DIR / 'xgb_classifier.pkl').exists(),
        (MODELS_DIR / 'best_threshold.pkl').exists(),
        (MODELS_DIR / 'xgb_regressor.pkl').exists(),
    ]
})
display(artifact_check)

print(f'Sample default probability from saved classifier: {sample_risk:.2%}')
print(f'Sample predicted SBA coverage: {sample_coverage:.2%}')
print(f'Sample fair-rate estimate: {sample_fair_rate:.1f}%')
print(f'Saved app threshold: {saved_threshold:.2f}')

assert artifact_check['exists'].all(), 'One or more app artifacts were not saved successfully.'
print('Artifact check passed: the notebook refreshed the files the app expects.')

## 9. Limitations and Future Work

### Limitations
- The historical file ends in **2014**, so the fair-rate baseline is not tied to modern prime-rate conditions.
- The dataset does not include every underwriting factor a real bank may use, such as collateral depth or personal guarantees.
- State-sector pockets can still be sparse. The app addresses this with support warnings, but rare combinations remain noisier than dense ones.
- The fair-rate output is a proxy built from predicted SBA guarantee behavior, not a direct supervised interest-rate label.

### Future Work
- connect the fair-rate formula to a live prime-rate feed
- add hierarchical smoothing for sparse state-sector combinations
- expand beyond two-digit sector codes toward more granular NAICS groupings
- retrain on newer SBA releases to improve modern relevance

## Dataset Citations (MLA 8)

Toktogaraev, Mirbek. *Should This Loan Be Approved or Denied?* Kaggle, https://www.kaggle.com/datasets/mirbektoktogaraev/should-this-loan-be-approved-or-denied. Accessed 29 Mar. 2026.

U.S. Small Business Administration. *7(a) & 504 FOIA*. SBA Open Data, https://data.sba.gov/en/dataset/7-a-504-foia. Accessed 29 Mar. 2026.